# GTEx-AHBA Core Vogel Recapitulation and Missing-Region Extrapolation

This notebook implements a Python-only workflow to:

1. Recapitulate core Vogel NB02/NB05-style AHBA spatial gradient findings.
2. Benchmark 4 classical methods to infer missing AHBA-covered brain regions in each GTEx subject.
3. Produce reproducible outputs under:
- `out/vogel_recap/`
- `out/extrapolation/`

The canonical in-notebook object is `bundle` with fixed keys:
`meta`, `genes_all`, `genes_hvg`, `ahba_df`, `gtex_df`, `coords`, `target_parcels`, `subject_observed_mask`, `domain_calibration`.

In [ ]:
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "statsmodels": "statsmodels",
}

def _has_module(mod_name: str) -> bool:
    try:
        __import__(mod_name)
        return True
    except Exception:
        return False

missing = [pkg for mod, pkg in required.items() if not _has_module(mod)]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", *missing])
else:
    print("All required packages are available.")


In [ ]:
import csv
import json
import math
import os
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.interpolate import RBFInterpolator
from scipy.spatial.transform import Rotation as R

from sklearn.base import clone
from sklearn.cross_decomposition import CCA, PLSCanonical, PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_context("notebook")
np.random.seed(123)

CONFIG = {
    "csv_path": "./gxp_samples.csv",
    "hvg_path": "./ahba_100hvg.txt",
    "output_vogel_dir": "./out/vogel_recap",
    "output_extrap_dir": "./out/extrapolation",
    "random_seed": 123,
    "n_components": 3,
    "n_pca": 100,
    "cv_splits": 5,
    "cv_repeats": 5,
    "n_permutations": 200,
    "n_rotations": 200,
    "gtex_holdout_frac": 0.25,
    "gtex_holdout_repeats": 5,
    "ahba_holdout_repeats": 5,
    "min_obs_for_cv": 6,
    "method2_latent_dim": 10,
    "method2_rbf_smoothing": 1.0,
    "method3_region_reg": 1e-3,
    "method3_blend_gtex": 0.3,
    "method4_rank": 10,
    "method4_ridge_alpha": 1.0,
    "run_stage2_all_genes": False,
    "stage2_chunk_size": 500,
    "stage2_top_k_methods": 2,
}

for p in [CONFIG["output_vogel_dir"], CONFIG["output_extrap_dir"]]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("Configured output directories:")
print(CONFIG["output_vogel_dir"])
print(CONFIG["output_extrap_dir"])

In [ ]:
COORD_PATTERN = re.compile(r"\(\s*([-\d\.eE]+)\s*,\s*([-\d\.eE]+)\s*,\s*([-\d\.eE]+)\s*\)")


def normalize_gene_name(name: str) -> str:
    return str(name).upper().replace("-", "_").replace(".", "_")


def parse_coordinate_centroid(coord_text: str):
    toks = COORD_PATTERN.findall(str(coord_text))
    if not toks:
        return (np.nan, np.nan, np.nan)
    arr = np.asarray([[float(a), float(b), float(c)] for a, b, c in toks], dtype=np.float64)
    return tuple(arr.mean(axis=0).tolist())


def load_gene_header_and_hvg(csv_path: str, hvg_path: str):
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

    meta_cols = header[:6]
    gene_cols = header[6:]

    with open(hvg_path) as f:
        hvg = [line.strip() for line in f if line.strip()]

    norm_map = {normalize_gene_name(g): g for g in gene_cols}
    matched_hvg = [norm_map[normalize_gene_name(g)] for g in hvg if normalize_gene_name(g) in norm_map]
    missing_hvg = [g for g in hvg if normalize_gene_name(g) not in norm_map]

    return {
        "meta_cols": meta_cols,
        "genes_all": gene_cols,
        "hvg_list": hvg,
        "genes_hvg": matched_hvg,
        "missing_hvg": missing_hvg,
    }


def read_expression_subset(csv_path: str, gene_cols):
    usecols = ["subject", "age", "sex", "dataset", "tissue_or_parcel", "coordinates"] + list(gene_cols)
    dtype_map = {
        "subject": "string",
        "age": "string",
        "sex": "string",
        "dataset": "string",
        "tissue_or_parcel": "string",
        "coordinates": "string",
    }
    dtype_map.update({g: np.float32 for g in gene_cols})
    df = pd.read_csv(csv_path, usecols=usecols, dtype=dtype_map, low_memory=False)
    return df


def add_coordinate_columns(df: pd.DataFrame):
    centroids = df["coordinates"].map(parse_coordinate_centroid)
    cent_df = pd.DataFrame(centroids.tolist(), columns=["coord_x", "coord_y", "coord_z"], index=df.index)
    out = pd.concat([df, cent_df], axis=1)
    out["coord_abs_x"] = out["coord_x"].abs()
    out["coord_valid"] = ~out[["coord_x", "coord_y", "coord_z"]].isna().any(axis=1)
    return out


def build_target_parcels(ahba_df: pd.DataFrame):
    grp = (
        ahba_df.groupby("tissue_or_parcel")[["coord_x", "coord_y", "coord_z"]]
        .mean()
        .reset_index()
        .sort_values("tissue_or_parcel")
        .reset_index(drop=True)
    )
    grp["parcel_idx"] = np.arange(len(grp), dtype=np.int32)
    return grp[["parcel_idx", "tissue_or_parcel", "coord_x", "coord_y", "coord_z"]]


def map_gtex_to_target(gtex_df: pd.DataFrame, target_parcels: pd.DataFrame):
    txyz = target_parcels[["coord_x", "coord_y", "coord_z"]].to_numpy(dtype=np.float64)
    tx = txyz[:, 0][:, None]
    ty = txyz[:, 1][:, None]
    tz = txyz[:, 2][:, None]

    gx = gtex_df["coord_x"].to_numpy(dtype=np.float64)[None, :]
    gy = gtex_df["coord_y"].to_numpy(dtype=np.float64)[None, :]
    gz = gtex_df["coord_z"].to_numpy(dtype=np.float64)[None, :]

    d2 = (tx - gx) ** 2 + (ty - gy) ** 2 + (tz - gz) ** 2
    idx = np.argmin(d2, axis=0)
    dist = np.sqrt(d2[idx, np.arange(d2.shape[1])])

    out = gtex_df.copy()
    out["parcel_idx"] = idx.astype(np.int32)
    out["mapped_parcel"] = target_parcels.loc[idx, "tissue_or_parcel"].to_numpy()
    out["mapping_distance"] = dist.astype(np.float32)
    return out


def _safe_std(vec: np.ndarray):
    sd = np.nanstd(vec, axis=0)
    sd = np.where(sd < 1e-8, 1.0, sd)
    return sd


def _parcel_gene_means(df: pd.DataFrame, gene_cols, n_parcels: int):
    G = len(gene_cols)
    out = np.full((n_parcels, G), np.nan, dtype=np.float64)
    for r in range(n_parcels):
        sub = df[df["parcel_idx"] == r]
        if len(sub) == 0:
            continue
        out[r, :] = sub[gene_cols].to_numpy(dtype=np.float64).mean(axis=0)
    return out


def fit_domain_calibration(ahba_raw: pd.DataFrame, gtex_raw: pd.DataFrame, gene_cols, n_parcels: int):
    X_a = ahba_raw[gene_cols].to_numpy(dtype=np.float64)
    X_g = gtex_raw[gene_cols].to_numpy(dtype=np.float64)

    ahba_mean = np.nanmean(X_a, axis=0)
    ahba_std = _safe_std(X_a)
    gtex_mean = np.nanmean(X_g, axis=0)
    gtex_std = _safe_std(X_g)

    ahba_z = (X_a - ahba_mean) / ahba_std
    gtex_z = (X_g - gtex_mean) / gtex_std

    ahba_tmp = ahba_raw.copy()
    gtex_tmp = gtex_raw.copy()
    ahba_tmp.loc[:, gene_cols] = pd.DataFrame(ahba_z.astype(np.float32), columns=gene_cols, index=ahba_tmp.index)
    gtex_tmp.loc[:, gene_cols] = pd.DataFrame(gtex_z.astype(np.float32), columns=gene_cols, index=gtex_tmp.index)

    A_parcel = _parcel_gene_means(ahba_tmp, gene_cols, n_parcels)
    G_parcel = _parcel_gene_means(gtex_tmp, gene_cols, n_parcels)

    slopes = np.ones(len(gene_cols), dtype=np.float64)
    intercepts = np.zeros(len(gene_cols), dtype=np.float64)

    for g in range(len(gene_cols)):
        x = G_parcel[:, g]
        y = A_parcel[:, g]
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() >= 2 and np.nanstd(x[m]) > 1e-8:
            b1, b0 = np.polyfit(x[m], y[m], deg=1)
            slopes[g] = np.clip(b1, -5.0, 5.0)
            intercepts[g] = b0
        elif m.sum() >= 1:
            slopes[g] = 1.0
            intercepts[g] = np.nanmean(y[m] - x[m])

    return {
        "genes": list(gene_cols),
        "ahba_mean": ahba_mean,
        "ahba_std": ahba_std,
        "gtex_mean": gtex_mean,
        "gtex_std": gtex_std,
        "slope": slopes,
        "intercept": intercepts,
    }


def apply_harmonization(df_raw: pd.DataFrame, gene_cols, calibration: dict, dataset_name: str):
    X = df_raw[gene_cols].to_numpy(dtype=np.float64)
    if dataset_name.upper() == "AHBA":
        X_h = (X - calibration["ahba_mean"]) / calibration["ahba_std"]
    else:
        Zg = (X - calibration["gtex_mean"]) / calibration["gtex_std"]
        X_h = Zg * calibration["slope"] + calibration["intercept"]

    out = df_raw.copy()
    out.loc[:, gene_cols] = pd.DataFrame(X_h.astype(np.float32), columns=gene_cols, index=out.index)
    return out


def build_subject_tensor(df: pd.DataFrame, gene_cols, n_parcels: int, subject_order=None):
    if subject_order is None:
        subjects = pd.Index(df["subject"].astype(str).unique())
    else:
        subjects = pd.Index(subject_order)

    sub_to_idx = {s: i for i, s in enumerate(subjects)}
    G = len(gene_cols)
    tensor = np.full((len(subjects), n_parcels, G), np.nan, dtype=np.float32)

    for _, row in df.iterrows():
        s = str(row["subject"])
        r = int(row["parcel_idx"])
        if r < 0 or r >= n_parcels:
            continue
        i = sub_to_idx.get(s)
        if i is None:
            continue
        vals = row[gene_cols].to_numpy(dtype=np.float32)
        if np.isnan(tensor[i, r]).all():
            tensor[i, r] = vals
        else:
            a = tensor[i, r]
            m = np.isfinite(a) & np.isfinite(vals)
            only_new = ~np.isfinite(a) & np.isfinite(vals)
            a[m] = 0.5 * (a[m] + vals[m])
            a[only_new] = vals[only_new]
            tensor[i, r] = a

    observed_mask = np.any(np.isfinite(tensor), axis=2)
    return tensor, subjects.to_numpy(), observed_mask


def build_bundle(csv_path: str, hvg_path: str, use_genes=None, fixed_target_parcels=None, scope_name="hvg"):
    header = load_gene_header_and_hvg(csv_path, hvg_path)
    genes_all = header["genes_all"]
    genes_hvg = header["genes_hvg"]

    if use_genes is None:
        gene_cols = genes_hvg
    else:
        gene_cols = list(use_genes)

    df = read_expression_subset(csv_path, gene_cols)
    df = add_coordinate_columns(df)
    df = df[df["coord_valid"]].copy()
    df["dataset_upper"] = df["dataset"].astype(str).str.upper().str.strip()

    ahba_raw = df[df["dataset_upper"] == "AHBA"].copy()
    gtex_raw = df[df["dataset_upper"] == "GTEX"].copy()

    if fixed_target_parcels is None:
        target_parcels = build_target_parcels(ahba_raw)
    else:
        target_parcels = fixed_target_parcels.copy().reset_index(drop=True)
        if "parcel_idx" not in target_parcels.columns:
            target_parcels["parcel_idx"] = np.arange(len(target_parcels), dtype=np.int32)

    parcel_lookup = dict(zip(target_parcels["tissue_or_parcel"], target_parcels["parcel_idx"]))
    ahba_raw["parcel_idx"] = ahba_raw["tissue_or_parcel"].map(parcel_lookup)
    ahba_raw = ahba_raw[ahba_raw["parcel_idx"].notna()].copy()
    ahba_raw["parcel_idx"] = ahba_raw["parcel_idx"].astype(np.int32)

    gtex_raw = map_gtex_to_target(gtex_raw, target_parcels)

    n_parcels = len(target_parcels)
    calibration = fit_domain_calibration(ahba_raw, gtex_raw, gene_cols, n_parcels=n_parcels)
    ahba_df = apply_harmonization(ahba_raw, gene_cols, calibration, "AHBA")
    gtex_df = apply_harmonization(gtex_raw, gene_cols, calibration, "GTEX")

    gtex_tensor, gtex_subjects, gtex_observed = build_subject_tensor(gtex_df, gene_cols, n_parcels=n_parcels)
    ahba_tensor, ahba_subjects, ahba_observed = build_subject_tensor(ahba_df, gene_cols, n_parcels=n_parcels)

    ahba_atlas = np.nanmean(ahba_tensor, axis=0)
    gtex_atlas = np.nanmean(gtex_tensor, axis=0)

    coords = {
        "target_xyz": target_parcels[["coord_x", "coord_y", "coord_z"]].to_numpy(dtype=np.float32),
        "ahba_xyz": ahba_df[["coord_x", "coord_y", "coord_z"]].to_numpy(dtype=np.float32),
        "gtex_xyz": gtex_df[["coord_x", "coord_y", "coord_z"]].to_numpy(dtype=np.float32),
    }

    bundle = {
        "meta": {
            "csv_path": csv_path,
            "hvg_path": hvg_path,
            "scope_name": scope_name,
            "gene_cols": list(gene_cols),
            "n_parcels": int(n_parcels),
            "row_meta": df[["subject", "age", "sex", "dataset", "tissue_or_parcel", "coordinates", "coord_x", "coord_y", "coord_z", "coord_abs_x"]].copy(),
            "missing_hvg": header["missing_hvg"],
        },
        "genes_all": genes_all,
        "genes_hvg": genes_hvg,
        "ahba_df": ahba_df,
        "gtex_df": gtex_df,
        "coords": coords,
        "target_parcels": target_parcels,
        "subject_observed_mask": gtex_observed,
        "domain_calibration": calibration,
        "gtex_tensor": gtex_tensor,
        "ahba_tensor": ahba_tensor,
        "ahba_atlas": ahba_atlas,
        "gtex_atlas": gtex_atlas,
        "gtex_subjects": gtex_subjects,
        "ahba_subjects": ahba_subjects,
        "ahba_subject_observed_mask": ahba_observed,
    }

    return bundle


bundle = build_bundle(CONFIG["csv_path"], CONFIG["hvg_path"], scope_name="hvg")

print("Bundle ready")
print("HVG matched:", len(bundle["genes_hvg"]))
print("AHBA rows:", len(bundle["ahba_df"]), "GTEx rows:", len(bundle["gtex_df"]))
print("Target parcels:", bundle["meta"]["n_parcels"])
print("GTEx subjects:", len(bundle["gtex_subjects"]))
print("AHBA donors:", len(bundle["ahba_subjects"]))
print("Missing HVG from file:", len(bundle["meta"]["missing_hvg"]))

In [ ]:
def map_gtex_division(tissue_name: str):
    t = str(tissue_name).strip().lower()
    mapping = {
        "brain - cortex": "Frontal",
        "brain - frontal cortex (ba9)": "Frontal",
        "brain - cerebellum": "Cerebellum",
        "brain - cerebellar hemisphere": "Cerebellum",
        "brain - caudate (basal ganglia)": "Caudate",
        "brain - nucleus accumbens (basal ganglia)": "Accumbens",
        "brain - putamen (basal ganglia)": "Putamen",
        "brain - hypothalamus": "Hypothalamus",
        "brain - hippocampus": "Hippocampus",
        "brain - anterior cingulate cortex (ba24)": "AntCing",
        "brain - substantia nigra": "Nigra",
        "brain - amygdala": "Amygdala",
    }
    return mapping.get(t, "Other")


def run_vogel_recap(bundle: dict, config: dict):
    out_dir = Path(config["output_vogel_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(config["random_seed"])

    ahba_df = bundle["ahba_df"].copy()
    gtex_df = bundle["gtex_df"].copy()
    genes = bundle["meta"]["gene_cols"]

    X = ahba_df[genes].to_numpy(dtype=np.float64)
    Y = ahba_df[["coord_y", "coord_z", "coord_abs_x"]].to_numpy(dtype=np.float64)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    n_pca = int(min(config["n_pca"], Xs.shape[0] - 1, Xs.shape[1]))
    pca = PCA(n_components=n_pca, random_state=config["random_seed"])
    Xp = pca.fit_transform(Xs)

    X_tr, X_te, y_tr, y_te = train_test_split(
        Xp,
        Y,
        test_size=0.25,
        random_state=config["random_seed"],
        shuffle=True,
    )

    candidates = {
        "PLSR": lambda nc: PLSRegression(n_components=nc),
        "CCA": lambda nc: CCA(n_components=nc),
        "PLSC": lambda nc: PLSCanonical(n_components=nc),
    }

    cv_rows = []
    for nc in range(1, config["n_components"] + 1):
        for name, ctor in candidates.items():
            est = ctor(nc)
            for rep in range(config["cv_repeats"]):
                cv = KFold(
                    n_splits=config["cv_splits"],
                    shuffle=True,
                    random_state=config["random_seed"] + rep,
                )
                pred = cross_val_predict(est, X_tr, y_tr, cv=cv)
                cv_rows.append(
                    {
                        "estimator": name,
                        "n_components": nc,
                        "repeat": rep,
                        "r2": float(r2_score(y_tr, pred, multioutput="variance_weighted")),
                        "mae": float(mean_absolute_error(y_tr, pred)),
                    }
                )

    cv_raw = pd.DataFrame(cv_rows)
    cv_raw.to_csv(out_dir / "estimator_selection_cv_raw.csv", index=False)
    cv_df = (
        cv_raw.groupby(["estimator", "n_components"], as_index=False)[["r2", "mae"]]
        .mean()
        .sort_values(["r2", "mae"], ascending=[False, True])
        .reset_index(drop=True)
    )
    cv_df.to_csv(out_dir / "estimator_selection_cv.csv", index=False)

    best = cv_df.iloc[0]

    pls = PLSRegression(n_components=config["n_components"])
    pls.fit(Xp, Y)
    test_pred = pls.predict(X_te)

    test_metrics = {
        "test_r2": float(r2_score(y_te, test_pred, multioutput="variance_weighted")),
        "test_mae": float(mean_absolute_error(y_te, test_pred)),
    }

    # permutation null on component score coupling
    real_comp = []
    for c in range(config["n_components"]):
        r = np.corrcoef(pls.x_scores_[:, c], pls.y_scores_[:, c])[0, 1]
        real_comp.append(float(r * r))

    null_comp = np.zeros((config["n_permutations"], config["n_components"]), dtype=np.float64)
    for i in range(config["n_permutations"]):
        perm_idx = rng.permutation(Xp.shape[0])
        perm_pls = PLSRegression(n_components=config["n_components"])
        perm_pls.fit(Xp[perm_idx], Y)
        for c in range(config["n_components"]):
            r = np.corrcoef(perm_pls.x_scores_[:, c], perm_pls.y_scores_[:, c])[0, 1]
            null_comp[i, c] = r * r

    perm_p = []
    for c in range(config["n_components"]):
        p = (np.sum(null_comp[:, c] >= real_comp[c]) + 1) / (len(null_comp) + 1)
        perm_p.append(float(p))

    # rotation null
    rot_scores = np.zeros((config["n_rotations"], config["n_components"]), dtype=np.float64)
    base_load = pls.x_loadings_.copy()
    for i in range(config["n_rotations"]):
        rotm = R.random(random_state=config["random_seed"] + i).as_matrix()
        Yr = Y @ rotm.T
        rpls = PLSRegression(n_components=config["n_components"])
        rpls.fit(Xp, Yr)
        for c in range(config["n_components"]):
            rr = np.corrcoef(base_load[:, c], rpls.x_loadings_[:, c])[0, 1]
            rot_scores[i, c] = rr * rr

    rot_p = []
    for c in range(config["n_components"]):
        p = (np.sum(rot_scores[:, c] >= real_comp[c]) + 1) / (len(rot_scores) + 1)
        rot_p.append(float(p))

    # GTEx projection
    Xg = gtex_df[genes].to_numpy(dtype=np.float64)
    Xg_p = pca.transform(scaler.transform(Xg))
    gtex_scores = pls.transform(Xg_p)
    ahba_scores = pls.transform(Xp)

    for c in range(config["n_components"]):
        gtex_df[f"C{c+1}"] = gtex_scores[:, c]
        ahba_df[f"C{c+1}"] = ahba_scores[:, c]

    gtex_df["Div"] = gtex_df["tissue_or_parcel"].map(map_gtex_division)

    # map AHBA parcels to division by GTEx majority vote among mapped GTEx rows
    parcel_major_div = (
        gtex_df.groupby("parcel_idx")["Div"].agg(lambda s: s.value_counts().index[0])
    )
    ahba_df["Div"] = ahba_df["parcel_idx"].map(parcel_major_div).fillna("Other")

    congr_rows = []
    for c in range(config["n_components"]):
        a = ahba_df.groupby("Div")[f"C{c+1}"].mean()
        g = gtex_df.groupby("Div")[f"C{c+1}"].mean()
        common = sorted(set(a.index) & set(g.index))
        if len(common) >= 3:
            rp, pp = stats.pearsonr(a.loc[common].values, g.loc[common].values)
            rs, ps = stats.spearmanr(a.loc[common].values, g.loc[common].values)
        else:
            rp = pp = rs = ps = np.nan
        congr_rows.append(
            {
                "component": c + 1,
                "n_divisions": len(common),
                "pearson_r": float(rp) if np.isfinite(rp) else np.nan,
                "pearson_p": float(pp) if np.isfinite(pp) else np.nan,
                "spearman_rho": float(rs) if np.isfinite(rs) else np.nan,
                "spearman_p": float(ps) if np.isfinite(ps) else np.nan,
            }
        )

    congr_df = pd.DataFrame(congr_rows)
    congr_df.to_csv(out_dir / "gtex_ahba_component_congruence.csv", index=False)

    # diagnostic plots
    fig, axs = plt.subplots(1, 3, figsize=(16, 4))
    pred_full = pls.predict(Xp)
    labels = ["y", "z", "|x|"]
    for i in range(3):
        axs[i].scatter(Y[:, i], pred_full[:, i], s=14, alpha=0.6)
        axs[i].set_xlabel(f"Observed {labels[i]}")
        axs[i].set_ylabel(f"Predicted {labels[i]}")
        axs[i].set_title(f"AHBA PLS coord fit ({labels[i]})")
    plt.tight_layout()
    plt.savefig(out_dir / "ahba_coord_fit_scatter.png", dpi=180)
    plt.close(fig)

    fig, axs = plt.subplots(1, config["n_components"], figsize=(5 * config["n_components"], 4))
    if config["n_components"] == 1:
        axs = [axs]
    for c in range(config["n_components"]):
        axs[c].hist(null_comp[:, c], bins=30, alpha=0.75, label="Permutation null")
        axs[c].axvline(real_comp[c], color="red", lw=2, label="Observed")
        axs[c].set_title(f"Comp {c+1}: p={perm_p[c]:.3g}")
        axs[c].set_xlabel("r^2(x_score, y_score)")
        axs[c].legend(loc="best")
    plt.tight_layout()
    plt.savefig(out_dir / "pls_permutation_null.png", dpi=180)
    plt.close(fig)

    summary = {
        "scope": bundle["meta"]["scope_name"],
        "n_ahba_samples": int(len(ahba_df)),
        "n_gtex_samples": int(len(gtex_df)),
        "n_genes": int(len(genes)),
        "n_components": int(config["n_components"]),
        "n_pca_components": int(n_pca),
        "best_estimator_cv": {
            "estimator": str(best["estimator"]),
            "n_components": int(best["n_components"]),
            "r2": float(best["r2"]),
            "mae": float(best["mae"]),
        },
        "test_metrics": test_metrics,
        "component_r2": real_comp,
        "component_perm_p": perm_p,
        "component_rot_p": rot_p,
        "gtex_ahba_congruence": congr_rows,
    }

    with open(out_dir / "vogel_recap_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    # persist reusable objects for downstream methods
    model_pack = {
        "scaler": scaler,
        "pca": pca,
        "pls": pls,
        "summary": summary,
        "cv_df": cv_df,
        "congruence_df": congr_df,
    }

    return model_pack


vogel_pack = run_vogel_recap(bundle, CONFIG)
print("Saved Vogel recap outputs to", CONFIG["output_vogel_dir"])

In [ ]:
def _bundle_gene_scope(bundle: dict, gene_scope: str):
    if gene_scope == "hvg":
        return bundle["gtex_tensor"], bundle["ahba_atlas"], bundle["meta"]["gene_cols"]
    if gene_scope == "all" and "gtex_tensor_all" in bundle and "ahba_atlas_all" in bundle:
        return bundle["gtex_tensor_all"], bundle["ahba_atlas_all"], bundle["genes_all"]
    raise ValueError(f"Unsupported gene_scope={gene_scope}. Available: hvg or prepared all")


def clone_bundle_with_tensor(bundle: dict, tensor: np.ndarray, observed_mask: np.ndarray, subjects=None):
    b = dict(bundle)
    b["gtex_tensor"] = tensor
    b["subject_observed_mask"] = observed_mask
    if subjects is not None:
        b["gtex_subjects"] = np.asarray(subjects)
    return b


def fit_method_1(bundle: dict, config: dict):
    coords = bundle["coords"]["target_xyz"].astype(np.float64)
    d2 = ((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2)
    return {
        "name": "atlas_delta_nearest_anchor",
        "ahba_atlas": bundle["ahba_atlas"].astype(np.float32),
        "dist2": d2.astype(np.float32),
    }


def predict_method_1(model: dict, bundle: dict, gene_scope: str = "hvg"):
    tensor, atlas, _ = _bundle_gene_scope(bundle, gene_scope)
    obs_mask = bundle["subject_observed_mask"]
    n_subj, R, G = tensor.shape
    pred = np.empty_like(tensor, dtype=np.float32)
    dist2 = model["dist2"]

    for s in range(n_subj):
        obs = np.where(obs_mask[s])[0]
        if len(obs) == 0:
            pred[s] = atlas
            continue

        pred[s] = atlas
        pred[s, obs, :] = tensor[s, obs, :]

        miss = np.where(~obs_mask[s])[0]
        for r in miss:
            o = obs[np.argmin(dist2[r, obs])]
            pred[s, r, :] = tensor[s, o, :] + (atlas[r, :] - atlas[o, :])

    return pred.astype(np.float32)


def fit_method_2(bundle: dict, config: dict):
    atlas = bundle["ahba_atlas"].astype(np.float64)
    Rn, Gn = atlas.shape
    latent_dim = int(min(config["method2_latent_dim"], Rn - 1, Gn, 20))

    scaler = StandardScaler()
    atlas_s = scaler.fit_transform(atlas)

    pca = PCA(n_components=latent_dim, random_state=config["random_seed"])
    Z = pca.fit_transform(atlas_s)

    decoder = Ridge(alpha=1e-2, random_state=config["random_seed"])
    decoder.fit(Z, atlas_s)

    fallback = fit_method_1(bundle, config)

    return {
        "name": "ahba_latent_spatial_rbf",
        "scaler": scaler,
        "pca": pca,
        "decoder": decoder,
        "latent_dim": latent_dim,
        "smoothing": float(config["method2_rbf_smoothing"]),
        "fallback": fallback,
    }


def predict_method_2(model: dict, bundle: dict, gene_scope: str = "hvg"):
    tensor, _, _ = _bundle_gene_scope(bundle, gene_scope)
    obs_mask = bundle["subject_observed_mask"]
    coords = bundle["coords"]["target_xyz"].astype(np.float64)

    n_subj, Rn, Gn = tensor.shape
    pred = np.empty((n_subj, Rn, Gn), dtype=np.float32)

    fallback_pred = predict_method_1(model["fallback"], bundle, gene_scope=gene_scope)

    for s in range(n_subj):
        obs = np.where(obs_mask[s])[0]
        if len(obs) < 2:
            pred[s] = fallback_pred[s]
            continue

        Y_obs = tensor[s, obs, :].astype(np.float64)
        Y_obs_s = model["scaler"].transform(Y_obs)
        Z_obs = model["pca"].transform(Y_obs_s)

        Z_full = np.zeros((Rn, model["latent_dim"]), dtype=np.float64)
        for c in range(model["latent_dim"]):
            try:
                rbf = RBFInterpolator(
                    coords[obs],
                    Z_obs[:, c],
                    smoothing=model["smoothing"],
                    kernel="thin_plate_spline",
                )
                Z_full[:, c] = rbf(coords)
            except Exception:
                nearest = np.argmin(((coords[:, None, :] - coords[obs][None, :, :]) ** 2).sum(axis=2), axis=1)
                Z_full[:, c] = Z_obs[nearest, c]

        Y_full_s = model["decoder"].predict(Z_full)
        Y_full = model["scaler"].inverse_transform(Y_full_s)

        pred[s] = Y_full.astype(np.float32)
        pred[s, obs, :] = tensor[s, obs, :]

    return pred


def _pairwise_cov_nan(X: np.ndarray):
    Rn = X.shape[1]
    C = np.zeros((Rn, Rn), dtype=np.float64)
    for i in range(Rn):
        xi = X[:, i]
        for j in range(i, Rn):
            xj = X[:, j]
            m = np.isfinite(xi) & np.isfinite(xj)
            if m.sum() > 1:
                c = np.cov(xi[m], xj[m], ddof=1)[0, 1]
            else:
                c = 0.0
            C[i, j] = C[j, i] = c
    return C


def fit_method_3(bundle: dict, config: dict):
    atlas = bundle["ahba_atlas"].astype(np.float64)
    gtex_tensor = bundle["gtex_tensor"].astype(np.float64)

    Sigma_ahba = np.cov(atlas, rowvar=True, ddof=1)
    region_signal = np.nanmean(gtex_tensor, axis=2)
    Sigma_gtex = _pairwise_cov_nan(region_signal)

    w = float(config["method3_blend_gtex"])
    Sigma = (1.0 - w) * Sigma_ahba + w * Sigma_gtex

    reg = float(config["method3_region_reg"])
    Sigma = Sigma + reg * np.eye(Sigma.shape[0], dtype=np.float64)

    return {
        "name": "conditional_gaussian_imputation",
        "ahba_atlas": atlas,
        "Sigma": Sigma,
        "reg": reg,
    }


def predict_method_3(model: dict, bundle: dict, gene_scope: str = "hvg"):
    tensor, atlas, _ = _bundle_gene_scope(bundle, gene_scope)
    obs_mask = bundle["subject_observed_mask"]

    n_subj, Rn, Gn = tensor.shape
    pred = np.empty((n_subj, Rn, Gn), dtype=np.float32)
    Sigma = model["Sigma"]

    for s in range(n_subj):
        obs = np.where(obs_mask[s])[0]
        miss = np.where(~obs_mask[s])[0]

        if len(obs) == 0:
            pred[s] = atlas
            continue

        pred[s] = atlas
        pred[s, obs, :] = tensor[s, obs, :]

        if len(miss) == 0:
            continue

        S_oo = Sigma[np.ix_(obs, obs)]
        S_mo = Sigma[np.ix_(miss, obs)]
        inv = np.linalg.pinv(S_oo)

        x_obs = tensor[s, obs, :].astype(np.float64)
        mu_obs = atlas[obs, :].astype(np.float64)
        mu_m = atlas[miss, :].astype(np.float64)

        delta = x_obs - mu_obs
        cond = mu_m + (S_mo @ inv @ delta)
        pred[s, miss, :] = cond.astype(np.float32)

    return pred


def fit_method_4(bundle: dict, config: dict):
    atlas = bundle["ahba_atlas"].astype(np.float64)
    centered = atlas - atlas.mean(axis=0, keepdims=True)

    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    rank = int(min(config["method4_rank"], centered.shape[0] - 1, centered.shape[1], len(S)))

    templates = np.zeros((rank, centered.shape[0], centered.shape[1]), dtype=np.float64)
    for k in range(rank):
        templates[k] = S[k] * np.outer(U[:, k], Vt[k, :])

    return {
        "name": "lowrank_ahba_prior_completion",
        "baseline": atlas,
        "templates": templates,
        "rank": rank,
        "ridge_alpha": float(config["method4_ridge_alpha"]),
    }


def predict_method_4(model: dict, bundle: dict, gene_scope: str = "hvg"):
    tensor, _, _ = _bundle_gene_scope(bundle, gene_scope)
    obs_mask = bundle["subject_observed_mask"]

    baseline = model["baseline"]
    templates = model["templates"]
    rank = model["rank"]
    alpha = model["ridge_alpha"]

    n_subj, Rn, Gn = tensor.shape
    pred = np.empty((n_subj, Rn, Gn), dtype=np.float32)

    for s in range(n_subj):
        obs = np.where(obs_mask[s])[0]
        if len(obs) == 0:
            pred[s] = baseline
            continue

        Y_obs = tensor[s, obs, :].astype(np.float64)
        B_obs = baseline[obs, :]
        y = (Y_obs - B_obs).reshape(-1)

        A = np.zeros((y.shape[0], rank), dtype=np.float64)
        for k in range(rank):
            A[:, k] = templates[k, obs, :].reshape(-1)

        AtA = A.T @ A + alpha * np.eye(rank)
        Aty = A.T @ y
        coef = np.linalg.solve(AtA, Aty)

        recon = baseline.copy()
        for k in range(rank):
            recon += coef[k] * templates[k]

        pred[s] = recon.astype(np.float32)
        pred[s, obs, :] = tensor[s, obs, :]

    return pred


def evaluate_predictions(pred_tensor: np.ndarray, truth_tensor: np.ndarray, mask: np.ndarray):
    y_true = truth_tensor[mask]
    y_pred = pred_tensor[mask]

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) < 2:
        return pd.DataFrame([
            {
                "n_points": int(len(y_true)),
                "pearson_r": np.nan,
                "spearman_rho": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "medae": np.nan,
            }
        ])

    pr = stats.pearsonr(y_true, y_pred)
    sr = stats.spearmanr(y_true, y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    medae = np.median(np.abs(y_true - y_pred))

    return pd.DataFrame([
        {
            "n_points": int(len(y_true)),
            "pearson_r": float(pr.statistic),
            "pearson_p": float(pr.pvalue),
            "spearman_rho": float(sr.statistic),
            "spearman_p": float(sr.pvalue),
            "rmse": float(rmse),
            "mae": float(mae),
            "medae": float(medae),
        }
    ])


def fit_all_methods(bundle: dict, config: dict):
    methods = {
        "method1": (fit_method_1(bundle, config), predict_method_1),
        "method2": (fit_method_2(bundle, config), predict_method_2),
        "method3": (fit_method_3(bundle, config), predict_method_3),
        "method4": (fit_method_4(bundle, config), predict_method_4),
    }
    return methods


def save_prediction_npz(path: Path, pred: np.ndarray, bundle: dict, method_name: str, scope: str):
    np.savez_compressed(
        path,
        predictions=pred.astype(np.float32),
        subjects=np.asarray(bundle["gtex_subjects"]),
        parcel_idx=np.asarray(bundle["target_parcels"]["parcel_idx"]),
        parcel_names=np.asarray(bundle["target_parcels"]["tissue_or_parcel"]),
        genes=np.asarray(bundle["meta"]["gene_cols"]),
        observed_mask=bundle["subject_observed_mask"],
        method=np.asarray([method_name]),
        scope=np.asarray([scope]),
    )


methods = fit_all_methods(bundle, CONFIG)
print("Fitted methods:", list(methods.keys()))

In [ ]:
# Full-scope HVG predictions for each method
extrap_dir = Path(CONFIG["output_extrap_dir"])
all_predictions = {}

for method_name, (model, predict_fn) in methods.items():
    pred = predict_fn(model, bundle, gene_scope="hvg")
    all_predictions[method_name] = pred
    out_path = extrap_dir / f"predictions_{method_name}_hvg.npz"
    save_prediction_npz(out_path, pred, bundle, method_name, "hvg")
    print("Saved", out_path)

# Save parcel mapping output
parcel_mapping = (
    bundle["gtex_df"][["tissue_or_parcel", "mapped_parcel", "parcel_idx", "mapping_distance"]]
    .drop_duplicates()
    .sort_values(["tissue_or_parcel", "mapping_distance"]) 
)
parcel_mapping.to_csv(extrap_dir / "parcel_mapping.csv", index=False)
print("Saved", extrap_dir / "parcel_mapping.csv")

In [ ]:
def run_gtex_pseudomissing_cv(bundle: dict, methods: dict, config: dict):
    rng = np.random.default_rng(config["random_seed"])

    truth = bundle["gtex_tensor"]
    obs_mask = bundle["subject_observed_mask"]
    n_subj, Rn, Gn = truth.shape

    obs_counts = obs_mask.sum(axis=1)
    eligible = np.where(obs_counts >= config["min_obs_for_cv"])[0]

    rows_by_method = {m: [] for m in methods}

    for rep in range(config["gtex_holdout_repeats"]):
        masked = truth.copy()
        eval_mask = np.zeros_like(masked, dtype=bool)

        for s in eligible:
            obs = np.where(obs_mask[s])[0]
            n_hold = max(1, int(math.ceil(len(obs) * config["gtex_holdout_frac"])))
            hold = rng.choice(obs, size=min(n_hold, len(obs) - 1), replace=False)
            masked[s, hold, :] = np.nan
            eval_mask[s, hold, :] = True

        masked_obs = np.any(np.isfinite(masked), axis=2)
        tmp_bundle = clone_bundle_with_tensor(bundle, masked, masked_obs)

        for method_name, (model, predict_fn) in methods.items():
            pred = predict_fn(model, tmp_bundle, gene_scope="hvg")
            met = evaluate_predictions(pred, truth, eval_mask)
            met.loc[:, "method"] = method_name
            met.loc[:, "split"] = "gtex_pseudomissing"
            met.loc[:, "repeat"] = rep
            met.loc[:, "n_subjects_eval"] = int(len(eligible))
            rows_by_method[method_name].append(met)

    out = {}
    for method_name, parts in rows_by_method.items():
        out[method_name] = pd.concat(parts, ignore_index=True)
    return out


gtex_cv = run_gtex_pseudomissing_cv(bundle, methods, CONFIG)

for method_name, dfm in gtex_cv.items():
    path = Path(CONFIG["output_extrap_dir"]) / f"metrics_{method_name}_gtex_pseudomissing.csv"
    dfm.to_csv(path, index=False)
    print("Saved", path)

In [ ]:
def run_ahba_pseudo_gtex(bundle: dict, methods: dict, config: dict):
    rng = np.random.default_rng(config["random_seed"] + 77)

    donor_tensor = bundle["ahba_tensor"]
    donor_mask = bundle["ahba_subject_observed_mask"]
    donor_ids = bundle["ahba_subjects"]

    gtex_obs_counts = bundle["subject_observed_mask"].sum(axis=1)
    gtex_obs_counts = gtex_obs_counts[gtex_obs_counts >= 2]
    if len(gtex_obs_counts) == 0:
        gtex_obs_counts = np.array([6], dtype=int)

    rows_by_method = {m: [] for m in methods}

    for rep in range(config["ahba_holdout_repeats"]):
        for d in range(donor_tensor.shape[0]):
            obs = np.where(donor_mask[d])[0]
            if len(obs) < 3:
                continue

            k_target = int(rng.choice(gtex_obs_counts))
            k_keep = max(2, min(k_target, len(obs) - 1))
            keep = np.sort(rng.choice(obs, size=k_keep, replace=False))
            hold = np.setdiff1d(obs, keep)

            pseudo = np.full((1, donor_tensor.shape[1], donor_tensor.shape[2]), np.nan, dtype=np.float32)
            pseudo[0, keep, :] = donor_tensor[d, keep, :]
            pseudo_obs = np.any(np.isfinite(pseudo), axis=2)

            eval_mask = np.zeros_like(pseudo, dtype=bool)
            eval_mask[0, hold, :] = True

            truth = donor_tensor[d:d+1]
            tmp_bundle = clone_bundle_with_tensor(bundle, pseudo, pseudo_obs, subjects=[f"AHBA_{donor_ids[d]}"])

            for method_name, (model, predict_fn) in methods.items():
                pred = predict_fn(model, tmp_bundle, gene_scope="hvg")
                met = evaluate_predictions(pred, truth, eval_mask)
                met.loc[:, "method"] = method_name
                met.loc[:, "split"] = "ahba_pseudo_gtex"
                met.loc[:, "repeat"] = rep
                met.loc[:, "donor"] = str(donor_ids[d])
                met.loc[:, "n_kept"] = int(k_keep)
                rows_by_method[method_name].append(met)

    out = {}
    for method_name, parts in rows_by_method.items():
        out[method_name] = pd.concat(parts, ignore_index=True)
    return out


ahba_cv = run_ahba_pseudo_gtex(bundle, methods, CONFIG)

for method_name, dfm in ahba_cv.items():
    path = Path(CONFIG["output_extrap_dir"]) / f"metrics_{method_name}_ahba_pseudo_gtex.csv"
    dfm.to_csv(path, index=False)
    print("Saved", path)

In [ ]:
def rank_methods(gtex_cv: dict, ahba_cv: dict):
    frames = []
    for d in [gtex_cv, ahba_cv]:
        for method, dfm in d.items():
            if len(dfm) == 0:
                continue
            agg = dfm.groupby(["method", "split"], as_index=False)[["pearson_r", "spearman_rho", "rmse", "mae", "medae"]].mean()
            frames.append(agg)

    allm = pd.concat(frames, ignore_index=True)

    score_rows = []
    for split, sdf in allm.groupby("split"):
        sdf = sdf.copy()
        sdf["rank_pearson"] = sdf["pearson_r"].rank(ascending=False, method="average")
        sdf["rank_spearman"] = sdf["spearman_rho"].rank(ascending=False, method="average")
        sdf["rank_rmse"] = sdf["rmse"].rank(ascending=True, method="average")
        sdf["rank_mae"] = sdf["mae"].rank(ascending=True, method="average")
        sdf["rank_medae"] = sdf["medae"].rank(ascending=True, method="average")
        sdf["split_rank_score"] = sdf[["rank_pearson", "rank_spearman", "rank_rmse", "rank_mae", "rank_medae"]].mean(axis=1)
        score_rows.append(sdf)

    ranked = pd.concat(score_rows, ignore_index=True)
    overall = ranked.groupby("method", as_index=False)["split_rank_score"].mean().rename(columns={"split_rank_score": "overall_rank_score"})
    overall = overall.sort_values("overall_rank_score").reset_index(drop=True)

    detailed = ranked.merge(overall, on="method", how="left").sort_values(["overall_rank_score", "split"]).reset_index(drop=True)
    return detailed, overall


rank_detailed, rank_overall = rank_methods(gtex_cv, ahba_cv)
rank_path = Path(CONFIG["output_extrap_dir"]) / "method_ranking.csv"
rank_detailed.to_csv(rank_path, index=False)
print("Saved", rank_path)

print("Overall method ranking:")
print(rank_overall)

In [ ]:
def iter_gene_chunks(genes, chunk_size):
    for start in range(0, len(genes), chunk_size):
        end = min(start + chunk_size, len(genes))
        yield start, end, genes[start:end]


def run_stage2_all_genes(bundle_hvg: dict, methods_rank_df: pd.DataFrame, config: dict):
    if not config["run_stage2_all_genes"]:
        print("Stage-2 all-gene chunked run is disabled (run_stage2_all_genes=False).")
        return

    top_methods = (
        methods_rank_df.sort_values("overall_rank_score")["method"]
        .drop_duplicates()
        .head(config["stage2_top_k_methods"])
        .tolist()
    )
    print("Stage-2 methods:", top_methods)

    csv_path = config["csv_path"]
    hvg_path = config["hvg_path"]
    genes_all = bundle_hvg["genes_all"]

    fixed_target = bundle_hvg["target_parcels"]
    out_dir = Path(config["output_extrap_dir"])

    for start, end, chunk_genes in iter_gene_chunks(genes_all, config["stage2_chunk_size"]):
        print(f"Processing all-gene chunk: {start}:{end} ({len(chunk_genes)} genes)")
        chunk_bundle = build_bundle(
            csv_path,
            hvg_path,
            use_genes=chunk_genes,
            fixed_target_parcels=fixed_target,
            scope_name=f"all_chunk_{start}_{end}",
        )

        chunk_methods = fit_all_methods(chunk_bundle, config)

        for method_name in top_methods:
            model, predict_fn = chunk_methods[method_name]
            pred = predict_fn(model, chunk_bundle, gene_scope="hvg")
            out_path = out_dir / f"predictions_{method_name}_all_chunk_{start}_{end}.npz"
            save_prediction_npz(out_path, pred, chunk_bundle, method_name, "all")
            print("Saved", out_path)


run_stage2_all_genes(bundle, rank_overall, CONFIG)

In [ ]:
# Final quick summary
print("\nWorkflow complete.")
print("Vogel outputs:", CONFIG["output_vogel_dir"])
print("Extrapolation outputs:", CONFIG["output_extrap_dir"])
print("Main summary file:", Path(CONFIG["output_vogel_dir"]) / "vogel_recap_summary.json")
print("Method ranking file:", Path(CONFIG["output_extrap_dir"]) / "method_ranking.csv")